In [1]:
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [2]:
MAX_SAMPLES = None
BATCH_SIZE  = 256
MIN_SUPPORT = 5

In [3]:
dataset = load_dataset("Roblox/RobloxGuard-Eval")
data = dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.76M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2873 [00:00<?, ? examples/s]

In [4]:
texts = [f"User: {p}\nAssistant: {r}" for p, r in zip(data["prompt"], data["response"])]
categories = ["None" if c in ("", None) else c for c in data["category"]]

n = MAX_SAMPLES or len(texts)
texts, true_labels = texts[:n], categories[:n]

counts = Counter(true_labels)
valid = {l for l, c in counts.items() if c >= MIN_SUPPORT}
pairs = [(t, l) for t, l in zip(texts, true_labels) if l in valid]
texts, true_labels = zip(*pairs)
texts, true_labels = list(texts), list(true_labels)

In [5]:
LABEL_DESCRIPTIONS = {
    "None":                                        "this conversation is safe and does not violate any rules",
    "Profanity":                                   "this conversation contains profanity or swear words",
    "Threats, Bullying, and Harassment":           "this conversation contains threats, bullying, or harassment of a person",
    "Romantic and Sexual Content":                 "this conversation contains romantic or sexual content",
    "Terrorism and Violent Extremism":             "this conversation promotes terrorism or violent extremism",
    "Illegal and Regulated Goods and Activities":  "this conversation discusses illegal goods, drugs, or regulated activities",
    "Sharing Personal Information":                "this conversation shares or requests personal information like address or phone number",
    "Child Exploitation":                          "this conversation exploits or endangers children",
    "Discrimination, Slurs, and Hate Speech":      "this conversation contains hate speech, slurs, or discrimination",
    "Violent Content and Gore":                    "this conversation contains graphic violence or gore",
    "Real-World Sensitive Events":                 "this conversation references sensitive real-world events like disasters or tragedies",
    "Political Figures and Entities":              "this conversation discusses political figures or political organizations",
    "Directing Users Off Platform":                "this conversation tries to direct users to external platforms or websites",
    "Suicide, Self Injury, and Harmful Behavior":  "this conversation discusses suicide, self-harm, or dangerous behavior",
    "Spam":                                        "this conversation contains spam or repetitive meaningless content",
    "Soliciting Donations":                        "this conversation solicits donations or asks for money",
    "Cheating and Scams":                          "this conversation involves cheating, scamming, or deceiving other users",
    "Misusing Roblox Systems":                     "this conversation describes misusing or exploiting Roblox platform systems",
    "Expanded Policies for Suitability":           "this conversation contains content unsuitable for the Roblox platform",
    "Intellectual Property Violations":            "this conversation contains intellectual property violations or copyright infringement",
    "Independent Advertisement Publishing":        "this conversation contains unauthorized or independent advertisements",
    "Prohibited Advertising Practices and Content":"this conversation contains prohibited advertising practices",
    "Paid Random Items":                           "this conversation involves paid random items or loot boxes",
}

EXISTING_LABELS  = sorted(set(true_labels))
LABEL_DESCS      = {k: v for k, v in LABEL_DESCRIPTIONS.items() if k in EXISTING_LABELS}
CATEGORY_LIST    = list(LABEL_DESCS.keys())
DESC_LIST        = list(LABEL_DESCS.values())

print(f"Examples   : {len(texts)}")
print(f"Categories : {len(CATEGORY_LIST)}")
for cat, count in Counter(true_labels).most_common():
    print(f"  {cat:<45} {count}")

Examples   : 2863
Categories : 19
  None                                          1981
  Illegal and Regulated Goods and Activities    124
  Romantic and Sexual Content                   99
  Real-World Sensitive Events                   97
  Terrorism and Violent Extremism               90
  Discrimination, Slurs, and Hate Speech        85
  Political Figures and Entities                81
  Directing Users Off Platform                  67
  Sharing Personal Information                  55
  Profanity                                     50
  Threats, Bullying, and Harassment             34
  Violent Content and Gore                      27
  Suicide, Self Injury, and Harmful Behavior    23
  Child Exploitation                            14
  Spam                                          13
  Expanded Policies for Suitability             8
  Soliciting Donations                          5
  Misusing Roblox Systems                       5
  Cheating and Scams                            

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("all-mpnet-base-v2", device=device)


Device: Tesla T4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
label_embeddings = model.encode(DESC_LIST, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

text_embeddings  = model.encode(
    [t[:2000] for t in texts],
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)


Embedding label descriptions...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding 2863 texts...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

In [8]:
sims  = text_embeddings @ label_embeddings.T
idxs  = sims.argmax(axis=1)
preds = [CATEGORY_LIST[i] for i in idxs]

print("Sample predictions:")
for i in range(5):
    print(f"  True: {true_labels[i]:<45} Pred: {preds[i]}")

confs = sims.max(axis=1).tolist()

print(f"\nPrediction distribution:")
for cat, count in Counter(preds).most_common():
    print(f"  {cat:<45} {count}")


Computing similarities...

Sample predictions:
  True: Spam                                          Pred: Expanded Policies for Suitability
  True: Terrorism and Violent Extremism               Pred: Terrorism and Violent Extremism
  True: Romantic and Sexual Content                   Pred: Violent Content and Gore
  True: Terrorism and Violent Extremism               Pred: Discrimination, Slurs, and Hate Speech
  True: Real-World Sensitive Events                   Pred: Violent Content and Gore

Prediction distribution:
  Romantic and Sexual Content                   248
  Suicide, Self Injury, and Harmful Behavior    244
  Misusing Roblox Systems                       219
  Expanded Policies for Suitability             210
  Political Figures and Entities                202
  None                                          169
  Violent Content and Gore                      161
  Terrorism and Violent Extremism               160
  Profanity                                     148
  D

In [9]:
all_label_names = sorted(set(true_labels) | set(preds))

acc = accuracy_score(true_labels, preds)
f1  = f1_score(true_labels, preds, average="macro", zero_division=0, labels=all_label_names)

print(f"\n{'='*55}")
print(f"  Results -- all-mpnet-base-v2 (zero-shot embedding)")
print(f"{'='*55}")
print(f"  Accuracy  : {acc:.4f}")
print(f"  F1 (macro): {f1:.4f}\n")
print(classification_report(true_labels, preds, labels=all_label_names, zero_division=0))


  Results -- all-mpnet-base-v2 (zero-shot embedding)
  Accuracy  : 0.1463
  F1 (macro): 0.1462

                                            precision    recall  f1-score   support

                        Cheating and Scams       0.01      0.20      0.02         5
                        Child Exploitation       0.03      0.21      0.05        14
              Directing Users Off Platform       0.16      0.28      0.20        67
    Discrimination, Slurs, and Hate Speech       0.19      0.32      0.24        85
         Expanded Policies for Suitability       0.00      0.12      0.01         8
Illegal and Regulated Goods and Activities       0.15      0.13      0.14       124
                   Misusing Roblox Systems       0.01      0.40      0.02         5
                                      None       0.81      0.07      0.13      1981
            Political Figures and Entities       0.14      0.35      0.20        81
                                 Profanity       0.21      0.6